> 在线讲义：[第 14 章 · nn.functional：归一化、激活与注意力](https://wikieden.github.io/pytorch-inside-out/lessons/14-functional-attention.html) · 本书《PyTorch 从里到外》

# 第 14 章 · `nn.functional`：归一化、激活与注意力

本章回答：LayerNorm 和 RMSNorm 差在哪？ReLU、GELU、SiLU 长什么样，SwiGLU 怎么拼？为什么不自己写 attention，而用 `scaled_dot_product_attention`？FlexAttention 又解决什么问题？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/14-functional-attention.md`（讲解、动画、练习都在讲义里）。对应原文：00-overview.md §4.6；AISPE ch13「PyTorch 优化注意力机制」（`ch13.md` 第 959–996 行）；官方文档 `F.scaled_dot_product_attention`、FlexAttention 博客（〔在线〕pytorch.org/blog/flexattention）；CS336 a1 handout §3（RMSNorm、SwiGLU）；本章输出在本机 torch 2.15.0 上跑过

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 归一化 | LN / RMSNorm | `norms()` | [−1.342, −0.447, 0.447, 1.342] / [0.365, 0.730, 1.095, 1.461] |
| 第三步 激活 | ReLU / GELU / SiLU | `activations()` | 见讲义表 |
| 第四步 SDPA | 和手写一致 | `sdpa_vs_manual()` | 最大差 1.6 × 10⁻⁷ |
| 第四步 分数表 | B·H·T²·2 | `scores_bytes(1, 32, 16384)` | 17.2 GB |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `activations`**（来自 `ch14_functional_attention.py`）：下面要用到的小工具。

In [ ]:
def activations():
    z = torch.tensor([-2.0, -0.5, 0.0, 1.0, 3.0])
    return F.relu(z), F.gelu(z), F.silu(z)

**函数 `norms`**（来自 `ch14_functional_attention.py`）：下面要用到的小工具。

In [ ]:
def norms():
    x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
    return F.layer_norm(x, (4,)), nn.RMSNorm(4)(x).detach()

**函数 `scores_bytes`**（来自 `ch14_functional_attention.py`）：手写 attention 的分数表字节数。

In [ ]:
def scores_bytes(B: int, H: int, T: int, bytes_per: int = 2) -> int:
    """手写 attention 的分数表字节数。"""
    return B * H * T * T * bytes_per

**函数 `manual_attention`**（来自 `ch14_functional_attention.py`）：下面要用到的小工具。

In [ ]:
def manual_attention(q, k, v):
    T, d = q.shape[-2], q.shape[-1]
    s = q @ k.transpose(-1, -2) / math.sqrt(d)
    s = s.masked_fill(torch.ones(T, T, dtype=torch.bool).triu(1), float("-inf"))
    return s.softmax(-1) @ v

**函数 `sdpa_vs_manual`**（来自 `ch14_functional_attention.py`）：下面要用到的小工具。

In [ ]:
def sdpa_vs_manual(seed: int = 0) -> float:
    torch.manual_seed(seed)
    q, k, v = (torch.randn(1, 2, 4, 8) for _ in range(3))
    return (F.scaled_dot_product_attention(q, k, v, is_causal=True) - manual_attention(q, k, v)).abs().max().item()

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 LayerNorm / RMSNorm：", norms())

In [ ]:
print("\n第三步 ReLU / GELU / SiLU：", activations())

In [ ]:
print("\n第四步 SDPA 与手写差：", sdpa_vs_manual())
print("分数表（B=1, H=32, T=16384, bf16）：", scores_bytes(1, 32, 16384) / 1e9, "GB")

## 3. 代码里才看得到的

1. **`nn.RMSNorm` 的输出带 `grad_fn`**：它有可训练的 γ，所以代码里 `.detach()` 了再打印。
2. **FlexAttention 在 Mac CPU 上编译不了**：本章代码只对比 SDPA；Flex 的不编译版本能跑但很慢。

## 4. 自己改一改

- 把 `norms` 的输入换成 [1, 1, 1, 1]，看 LayerNorm 输出什么（注意 ε）。
- `scores_bytes` 换成 T = 4096 和 32768，看比例。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 有参数用 nn，纯计算用 F；LayerNorm 减均值除标准差，RMSNorm 只除均方根；
> GELU、SiLU 是平滑 ReLU，SwiGLU 门乘线性再投影；
> attention 用 SDPA，自动挑 Flash 不写 T 方表；想改规则写小函数，Flex 编成 kernel。